# Exploratory Data Analysis: Candidates Dataset

This notebook performs the initial Exploratory Data Analysis (EDA) on the `candidates.csv` dataset. The objective is to understand the raw data, identify any anomalies or quality issues, apply a specific business rule for hiring, validate our Data Contract, and generate a Transformation Roadmap for our ETL pipeline.

## Data Contract Expectation
Before diving into the data, we expect:
- **Identifier/Keys**: First Name, Last Name, and Email to be present. Email must be valid (not strictly checked here, but expected non-null).
- **Categoricals**: `Seniority`, `Technology`, and `Country` must be standardized string values without nulls.
- **Numerics**: `YOE` (Years of Experience) must be a non-negative integer ($\ge 0$). `Code Challenge Score` and `Technical Interview Score` must be integers between $0$ and $10$.
- **Temporal**: `Application Date` should be a valid date in the past (up to 2022, per our analysis context).
- **No Duplicates**: Each application should be unique.


In [19]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load the data. Has a non-standard separator (semicolons).
file_path = '../data/raw/candidates.csv'
df = pd.read_csv(file_path, sep=';')

# Display shape and first few rows
print(f"Dataset Shape: {df.shape}")
display(df.head())


Dataset Shape: (50000, 10)


,First Name,Last Name,Email,Application Date,Country,YOE,Seniority,Technology,Code Challenge Score,Technical Interview Score
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9
3,Arch,Spinka,elvera_kulas@yahoo.com,2020-10-01,Eritrea,25,Trainee,QA Manual,7,1
4,Larue,Altenwerth,minnie.gislason@gmail.com,2020-05-20,Myanmar,13,Mid-Level,Social Media Community Management,9,7


## 1. Data Quality Issues (Missing Values & Duplicates)
**Reasoning & Expectations**: 
We must ensure there are no missing values (Nulls/NaNs) in critical columns. If there are, we need a strategy to handle them. We also need to check for complete duplicate rows which could skew analytics.

**Contract**: 
- `Null count` should be 0 across all columns.
- `Duplicated count` should be 0.


In [20]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found.")

# Check for exact duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicates}")


Missing Values per Column:
No missing values found.

Duplicate Rows: 0


## 2. Numerical Anomalies
**Reasoning & Expectations**: 
`YOE` (Years of Experience) cannot be negative and should be within realistic human bounds (e.g., $0-50$ years). Both `Code Challenge Score` and `Technical Interview Score` should be strictly between $0$ and $10$.

**Contract**: 
- $YOE \ge 0$
- $0 \le Scores \le 10$


In [21]:
numeric_cols = ['YOE', 'Code Challenge Score', 'Technical Interview Score']
display(df[numeric_cols].describe().T)

# Highlighting specific violations
invalid_yoe = df[df['YOE'] < 0]
invalid_scores = df[
    (df['Code Challenge Score'] < 0) | (df['Code Challenge Score'] > 10) |
    (df['Technical Interview Score'] < 0) | (df['Technical Interview Score'] > 10)
]

print(f"Records with Negative YOE: {len(invalid_yoe)}")
print(f"Records with Invalid Scores (<0 or >10): {len(invalid_scores)}")


,count,mean,std,min,25%,50%,75%,max
YOE,50000.0,15.28698,8.830652,0.0,8.0,15.0,23.0,30.0
Code Challenge Score,50000.0,4.99640,3.166896,0.0,2.0,5.0,8.0,10.0
Technical Interview Score,50000.0,5.00388,3.165082,0.0,2.0,5.0,8.0,10.0


Records with Negative YOE: 0
Records with Invalid Scores (<0 or >10): 0


## 3. Categorical Health
**Reasoning & Expectations**: 
We need to understand the distribution of `Seniority`, `Technology`, and `Country`. We look for typos, inconsistent casing, or unexpected categories that require mapping (e.g., 'USA' vs 'United States').

**Contract**: 
- Values should be consistent.
- `Seniority` should map to known levels (Intern, Junior, Mid-Level, Senior, Lead, Architect, etc.)


In [22]:
categories = ['Seniority', 'Technology']

for col in categories:
    print(f"\n--- {col} Distribution ---")
    print(df[col].value_counts())

print(f"\n--- Country Sample (Top 10) ---")
print(df['Country'].value_counts().head(10))



--- Seniority Distribution ---
Seniority
Intern       7255
Mid-Level    7253
Trainee      7183
Junior       7100
Architect    7079
Lead         7071
Senior       7059
Name: count, dtype: int64

--- Technology Distribution ---
Technology
Game Development                           3818
DevOps                                     3808
Social Media Community Management          2028
System Administration                      2014
Mulesoft                                   1973
Development - Backend                      1965
Development - FullStack                    1961
Adobe Experience Manager                   1954
Data Engineer                              1951
Security                                   1936
Development - CMS Frontend                 1934
Business Intelligence                      1934
Database Administration                    1933
Client Success                             1927
Design                                     1906
QA Manual                                 

## 4. Temporal Logic
**Reasoning & Expectations**: 
`Application Date` should be a valid Date string and convertible to a datetime object without errors.

**Contract**: 
- Proper format matching `YYYY-MM-DD`.
- No future dates beyond the current context.


In [23]:
# Attempt conversion to datetime
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')

# Check for coercion failures (NaT)
invalid_dates = df['Application Date'].isnull().sum()
print(f"Invalid/Unparseable Dates: {invalid_dates}")

if invalid_dates == 0:
    print(f"Date Range: {df['Application Date'].min().date()} to {df['Application Date'].max().date()}")


Invalid/Unparseable Dates: 0
Date Range: 2018-01-01 to 2022-07-04


## 5. Business Rule Implementation
**Rule**: A candidate is considered **Hired** if: `Code Challenge Score >= 7 AND Technical Interview Score >= 7`.

**Expectation**: We will test this logic and evaluate the distribution of Hired vs Not Hired candidates.


In [24]:
# Apply Hired rule
df['Hired'] = (df['Code Challenge Score'] >= 7) & (df['Technical Interview Score'] >= 7)

# Analyze Hiring Distribution
hiring_dist = df['Hired'].value_counts(normalize=True) * 100
print("Hiring Distribution (%):")
print(hiring_dist)

print(f"\nTotal Candidates Hired: {df['Hired'].sum()}")


Hiring Distribution (%):
Hired
False    86.604
True     13.396
Name: proportion, dtype: float64

Total Candidates Hired: 6698


## 6. Data Contract Validation
**Reasoning & Expectations**: 
A "clean" record is explicitly defined as one that satisfies all our constraints. Any record violating these constraints must be flagged.

**Contract rules applied**:
1. No Nulls in required columns.
2. `YOE` $\ge 0$ AND `YOE` $\le 50$.
3. `Scores` between $0$ and $10$.


In [25]:
# Flagging violations
df['is_valid'] = (
    df.notnull().all(axis=1) & 
    (df['YOE'] >= 0) & (df['YOE'] <= 50) & 
    (df['Code Challenge Score'].between(0, 10)) & 
    (df['Technical Interview Score'].between(0, 10))
)

valid_count = df['is_valid'].sum()
invalid_count = len(df) - valid_count

print(f"Valid Records: {valid_count} ({(valid_count/len(df))*100:.2f}%)")
print(f"Invalid Records: {invalid_count} ({(invalid_count/len(df))*100:.2f}%)")

if invalid_count > 0:
    display(df[~df['is_valid']].head())


Valid Records: 50000 (100.00%)
Invalid Records: 0 (0.00%)


## 7. Transformation Roadmap
Based on our EDA discoveries, the data is exceptionally clean (`0` nulls, `0` duplicates, and all numeric constraints are satisfied implicitly). However, the ETL pipeline must still enforce these operations to ensure production safety and alignment with the Star Schema design:

### Roadmap for `src/transform.py`

(btw, this roadmap is clrearly improoved with the help of AI, in this point of the work i want/wanted to have a clear vision on how to proceed with the extraction step)

1. **Extraction & Typing**:
    - Load the raw data setting `sep=';'`.
    - Enforce types explicitly (e.g., Cast `Application Date` to `datetime`, `YOE`, `Scores` to `int`).

2. **Validation & Cleansing (Safety Nets)**:
    - Drop or flag duplicate rows.
    - Handle potential nulls (using `.dropna()` or imputation/default values).
    - Enforce bounds: Filter out `YOE < 0` or scores `<0` | `>10`.

3. **Business Logic Injection**:
    - Create the `Hired` boolean/integer column based on: `(Code Challenge Score >= 7) & (Technical Interview Score >= 7)`.

4. **Dimensional Modeling Transformations (Star Schema Setup)**:
    - **Date Dimension (`dim_times`)**: Extract `Year`, `Month`, `Day`, and `Quarter` from `Application Date` to populate the dimension and generate a surrogate `date_id`.
    - **Candidate Dimension (`dim_candidates`)**: Isolate `First Name`, `Last Name`, and `Email`, generate a unique surrogate `candidate_id`.
    - **Job Dimension (`dim_jobs`)**: Isolate unique combinations of `Technology` and `Seniority`, generate a surrogate `job_id`.
    - **Location Dimension (`dim_locations`)**: Isolate unique `Country` values, generate a surrogate `location_id`.
    - **Fact Table Generation (`fact_applications`)**: Assemble the final table by mapping all records to their respective surrogate keys (`candidate_id`, `date_id`, `job_id`, `location_id`) and include a unique `application_id` as the Primary Key, alongside measures: `YOE`, scores, and the `is_hired` flag.

This strategy strictly aligns with the Data Contract validated here and perfectly bridges raw data to the final Dimensional Model required for KPIs.

Disclaimer: I used AI to refine the comments on the markdowns, and also to make some functions that i alone would not be able to do (at least not as good as i would like) Like, for example, the last cell Data Contract Validation.